In [9]:
import os, re, glob
import numpy as np
import pandas as pd

# ---------- configuration ----------
systems = {
    "WTiV":  "WTiV",   # folder path for W-Ti-V CSVs
    "WTiZr": "WTiZr",  # folder path for W-Ti-Zr CSVs
    "WVZr":  "WVZr",   # folder path for W-V-Zr CSVs
}

file_pattern = "*.csv"         # expects 1.csv … 231.csv in each folder
targets_fL = (0.60, 0.10, 0.01)
output_excel = "CSC_all_systems.xlsx"
# -----------------------------------

def read_tc_csv(path):
    """Read a Thermo-Calc CSV and return (fL, Q) arrays sorted by fL."""
    df = pd.read_csv(path, sep=None, engine="python")
    orig_cols = df.columns
    norm_cols = (orig_cols
                 .str.replace('\u00A0', ' ', regex=False)
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip()
                 .str.lower())
    norm2orig = dict(zip(norm_cols, orig_cols))

    try:
        norm_fL = next(c for c in norm_cols if re.search(r'\bmole\b.*\bfraction\b.*\bliquid\b', c))
        norm_Q  = next(c for c in norm_cols if re.search(r'\bheat\b.*\bper\b.*\bmol', c))
    except StopIteration:
        raise ValueError(f"Could not detect required columns in {path}")

    col_fL, col_Q = norm2orig[norm_fL], norm2orig[norm_Q]
    f = pd.to_numeric(df[col_fL], errors='coerce')
    Q = pd.to_numeric(df[col_Q],  errors='coerce')
    m = f.notna() & Q.notna()
    f, Q = f[m].to_numpy(), Q[m].to_numpy()
    order = np.argsort(f)
    return f[order], Q[order]

def interp_Q_at_fL(f, Q, target):
    """Linear interpolation of Q at given fL target (NO extrapolation)."""
    idx = np.where((f[:-1] <= target) & (f[1:] >= target) |
                   (f[:-1] >= target) & (f[1:] <= target))[0]
    if len(idx) == 0:
        raise ValueError(f"Target fL={target} outside range [{f.min():.3f}, {f.max():.3f}]")
    i = idx[0]
    w = (target - f[i]) / (f[i+1] - f[i])
    return Q[i] + w*(Q[i+1]-Q[i])

def compute_csc(f, Q):
    """Compute CSC using the dQ/sqrt(t) assumption."""
    Q06 = interp_Q_at_fL(f, Q, 0.60)
    Q01 = interp_Q_at_fL(f, Q, 0.10)
    Q001 = interp_Q_at_fL(f, Q, 0.01)
    dQ01, dQ001 = abs(Q01 - Q06), abs(Q001 - Q06)
    tR = dQ01**2
    tv = dQ001**2 - dQ01**2
    return Q06, Q01, Q001, tv / tR

# --- NEW: temperature-only reader and mushy zone from extents (no interpolation) ---

def read_temp_columns(path):
    """
    Read 'Mole fraction of liquid' and 'Temperature [K]' only.
    Returns (fL_sorted, T_sorted) sorted by fL (ascending).
    """
    df = pd.read_csv(path, sep=None, engine="python")

    # find liquid-fraction column robustly
    orig_cols = df.columns
    norm_cols = (orig_cols
                 .str.replace('\u00A0',' ', regex=False)
                 .str.replace(r'\s+',' ', regex=True)
                 .str.strip().str.lower())
    norm2orig = dict(zip(norm_cols, orig_cols))
    try:
        norm_fL = next(c for c in norm_cols if re.search(r'\bmole\b.*\bfraction\b.*\bliquid\b', c))
    except StopIteration:
        raise ValueError("Could not detect 'Mole fraction of liquid' column")

    # temperature column (you said it is named exactly this)
    if "Temperature [K]" not in df.columns:
        raise ValueError("'Temperature [K]' column not found")

    fL = pd.to_numeric(df[norm2orig[norm_fL]], errors='coerce')
    T  = pd.to_numeric(df["Temperature [K]"], errors='coerce')

    m = fL.notna() & T.notna()
    fL, T = fL[m].to_numpy(), T[m].to_numpy()

    # sort by fL ascending so min is index 0 and max is last
    order = np.argsort(fL)
    return fL[order], T[order]

def compute_mushy_from_extents(fL_sorted, T_sorted):
    """
    Take TL as T at max fL (closest to fL=1),
    TS as T at min fL (lowest available),
    ΔT = TL - TS. No interpolation/extrapolation.
    """
    if fL_sorted.size == 0:
        return None, None, None
    TS = T_sorted[0]          # temperature at minimum fL
    TL = T_sorted[-1]         # temperature at maximum fL
    dT = TL - TS
    return TL, TS, dT

# ---- Process all systems ----
all_systems = {}

for sys_name, folder in systems.items():
    rows = []
    paths = sorted(glob.glob(os.path.join(folder, file_pattern)),
                   key=lambda p: (len(os.path.basename(p)), os.path.basename(p)))
    for path in paths:
        comp_id = os.path.splitext(os.path.basename(path))[0]
        row = {"System": sys_name, "Composition_ID": comp_id, "File": path}
        try:
            # CSC path (unchanged)
            f, Q = read_tc_csv(path)
            Q06, Q01, Q001, CSC = compute_csc(f, Q)
            row.update({
                "Q@fL=0.60": Q06,
                "Q@fL=0.10": Q01,
                "Q@fL=0.01": Q001,
                "CSC": CSC
            })
        except Exception as e:
            row.update({"Q@fL=0.60": None, "Q@fL=0.10": None,
                        "Q@fL=0.01": None, "CSC": None, "CSC_Error": str(e)})

        # Mushy-zone temps (independent of CSC; no interpolation)
        try:
            fL_T, T = read_temp_columns(path)
            TL, TS, dT = compute_mushy_from_extents(fL_T, T)
            row.update({"TL [K]": TL, "TS [K]": TS, "ΔT [K]": dT})
        except Exception as e:
            row.update({"TL [K]": None, "TS [K]": None, "ΔT [K]": None, "T_Error": str(e)})

        rows.append(row)
    all_systems[sys_name] = pd.DataFrame(rows)

# ---- Save all to Excel with separate sheets ----
with pd.ExcelWriter(output_excel, engine="xlsxwriter") as writer:
    for sys_name, df_sys in all_systems.items():
        df_sys.to_excel(writer, sheet_name=sys_name, index=False)
    # optional combined sheet
    pd.concat(all_systems.values(), ignore_index=True).to_excel(writer, sheet_name="ALL", index=False)

print(f"\n✅ Done! CSC kept as-is. TL/TS/ΔT added using T at max/min fL (no interpolation).")



✅ Done! CSC kept as-is. TL/TS/ΔT added using T at max/min fL (no interpolation).
